# GRACE H3-v2 — CPU/RAM trên Colab

Chọn **Runtime → Change runtime type → Hardware accelerator: None**. Đây là phần CPU: không gọi LLM, không fine-tune CodeT5.

Chạy hai lần, đều bắt đầu từ notebook full-pipeline.ipynb local:

- **PHASE='base'**: upload base-inputs.zip + runtime → fit classifier/nested OOF, prepare validation → base-output.zip.
- **PHASE='meta'**: sau khi local đã chạy GRACE validation, upload meta-inputs.zip + runtime → fit selector/router/stacking, khóa model, tính test scores và prepare test → meta-output.zip.

Bundles chứa model joblib: chỉ import zip do project của bạn tạo. Google Drive giữ checkpoint; resume giữ các fit hoàn chỉnh, fit đang chạy có thể phải làm lại.


In [ ]:
from pathlib import Path
import os,sys,json,zipfile,hashlib,shutil,subprocess,time
PHASE='base'  # Lần 2: đổi thành 'meta'
USE_GOOGLE_DRIVE=True
DOWNLOAD_OUTPUT=False
if PHASE not in ('base','meta'): raise ValueError('PHASE phải là base hoặc meta')
ROOT=Path('/content/GRACE-H3-BenchmarkEnsemble')
ROOT.mkdir(parents=True,exist_ok=True)
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    TRANSFER=Path('/content/drive/MyDrive/GRACE-H3-v2-transfer')
else:
    TRANSFER=Path('/content/h3v2-transfer')
TRANSFER.mkdir(parents=True,exist_ok=True)
INPUT=TRANSFER/f'grace-h3v2-{PHASE}-inputs.zip'
RUNTIME=TRANSFER/'grace-h3-joern-runtime.zip'
CHECKPOINT=TRANSFER/f'grace-h3v2-{PHASE}-checkpoint.zip'
if not INPUT.exists() or not RUNTIME.exists():
    from google.colab import files
    print('Upload:',INPUT.name,'và',RUNTIME.name)
    for name,content in files.upload().items():
        if name not in (INPUT.name,RUNTIME.name,CHECKPOINT.name): raise ValueError('Tên file không thuộc phase hiện tại')
        (TRANSFER/name).write_bytes(content)
if not INPUT.exists() or not RUNTIME.exists(): raise FileNotFoundError('Thiếu input/runtime zip')
print('PHASE:',PHASE,'ROOT:',ROOT,'TRANSFER:',TRANSFER)


## Bootstrap source và nhập input

Bootstrap chỉ đưa source/config/requirements đã kiểm tra checksum vào workspace. Sau đó importer kiểm tra toàn bộ archive trước khi publish data/model. Hai phase có schema riêng để tránh nhập nhầm output cũ.


In [ ]:
with zipfile.ZipFile(INPUT) as z:
    m=json.loads(z.read('bundle_manifest.json'))
    if (m.get('schema'),m.get('kind'),m.get('phase'))!=('h3v2_bundle_v1','inputs',PHASE): raise ValueError('Sai input bundle/phase')
    entries=m['entries']
    names=[e['path'] for e in entries]
    if len({n.casefold() for n in names})!=len(names) or sorted(z.namelist())!=sorted(names+['bundle_manifest.json']): raise ValueError('Archive members không hợp lệ')
    for e in entries:
        name=e['path'];p=Path(name)
        if p.is_absolute() or '..' in p.parts or ':' in name or chr(92) in name: raise ValueError('Unsafe path')
        if not (name.startswith(('grace_h1/','grace_h3v2/','configs/')) or name.startswith('requirements')): continue
        target=ROOT/name;target.resolve().relative_to(ROOT.resolve())
        content=z.read(name)
        if len(content)!=e['bytes'] or hashlib.sha256(content).hexdigest()!=e['sha256']: raise ValueError('Source checksum mismatch')
        if target.exists() and target.read_bytes()!=content: raise ValueError('Workspace source khác input; dùng runtime/workspace mới')
        target.parent.mkdir(parents=True,exist_ok=True)
        target.write_bytes(content)
sys.path.insert(0,str(ROOT))
from grace_h3v2.portable import import_bundle,create_bundle
print(import_bundle(ROOT,INPUT,'inputs',PHASE))
print(import_bundle(ROOT,RUNTIME,'joern_runtime',PHASE))
if CHECKPOINT.exists(): print(import_bundle(ROOT,CHECKPOINT,'checkpoint',PHASE))


## Dependencies CPU và Joern

Ghim cùng phiên bản NumPy/SciPy/sklearn/joblib với local để load model ổn định. Java21 dùng runtime Joern đã đóng gói. Chỉ artifacts/cache/dataset đi qua bundle; credentials ở local.

Nếu phiên bản package bị cài mới/thay đổi, notebook sẽ dừng để bạn chọn **Runtime → Restart session**, rồi Run All lại từ đầu. Input và checkpoint trên Drive vẫn còn.


In [ ]:
from importlib.metadata import version, PackageNotFoundError
runtime_packages=('numpy','scipy','scikit-learn','joblib','threadpoolctl','PyYAML','rapidfuzz','python-dotenv','pandas')
def _runtime_versions():
    result={}
    for package in runtime_packages:
        try: result[package]=version(package)
        except PackageNotFoundError: result[package]=None
    return result
versions_before=_runtime_versions()
subprocess.check_call([sys.executable,'-m','pip','install','-r',str(ROOT/'requirements-v2.txt')])
versions_after=_runtime_versions()
if versions_before!=versions_after:
    raise RuntimeError('CPU packages were installed/updated. Select Runtime → Restart session, then run this notebook again from the first cell. The inputs and checkpoints on Google Drive are preserved.')
java_home=Path('/usr/lib/jvm/java-21-openjdk-amd64')
if not (java_home/'bin/java').exists():
    subprocess.check_call(['apt-get','update','-qq'])
    subprocess.check_call(['apt-get','install','-y','openjdk-21-jre-headless'])
if not (java_home/'bin/java').exists(): raise FileNotFoundError('Cần Java21; kiểm tra JAVA_HOME runtime')
os.environ['JAVA_HOME']=str(java_home)
os.environ['PATH']=str(java_home/'bin')+os.pathsep+os.environ['PATH']
os.environ['JOERN_HOME']=str(ROOT/'tools/joern/joern-cli')
for name in ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS'): os.environ[name]='4'
from threadpoolctl import threadpool_limits
threadpool_limits(limits=4)
from grace_h3v2 import workflow as w
CFG=w.config(ROOT)
print('Datasets:',CFG['datasets'],'seeds:',CFG['seeds'])
print('Main objective:',CFG['selection'])


## Fit và checkpoint

Base: nested train OOF, vocabulary/scaler fit riêng trong fold, fusion học từ OOF trong outer train; không chọn classifier bằng test.

Meta: classifier được khóa, GRACE validation nằm ngoài test, meta-model dùng3fold OOF để chọn recipe/cutoff rồi refit eligible validation. Validation đã dùng học meta-model nên không gọi điểm refit là independent validation.

Checkpoint được tạo sau5fit hoàn chỉnh hoặc khi ít nhất5phút đã trôi qua tại callback; graph callback mỗi500record. Nếu Interrupt chờ executor Joern hoàn thành các task đang submit thì checkpoint có thể trễ. Không tắt runtime trước khi checkpoint đã copy Drive xong.


In [ ]:
last_checkpoint={'time':time.monotonic(),'events':0}
def save_transfer(path):
    target=TRANSFER/path.name
    if path.resolve()!=target.resolve():
        temporary=target.with_suffix(target.suffix+'.tmp')
        shutil.copyfile(path,temporary)
        os.replace(temporary,target)
    return target
def checkpoint(force=False):
    last_checkpoint['events']+=1
    due=last_checkpoint['events']%5==0 or time.monotonic()-last_checkpoint['time']>=300
    if not (force or due): return
    path=ROOT/'bundles'/CHECKPOINT.name
    print(create_bundle(ROOT,path,'checkpoint',PHASE))
    print('Checkpoint:',save_transfer(path))
    last_checkpoint['time']=time.monotonic()
try:
    result=w.base_prepare(ROOT,CFG,checkpoint) if PHASE=='base' else w.meta_prepare(ROOT,CFG,checkpoint)
finally:
    checkpoint(force=True)
print(json.dumps(result,indent=2))


## Xuất kết quả và quay về entry local

Lấy output zip về **bundles/** trong worktree GRACE-H3-BenchmarkEnsemble rồi Run All full-pipeline.ipynb:

- base-output → local preflight/inference validation → tạo meta-inputs.
- meta-output → local preflight/inference test → predict/evaluate.

Chưa có điểm test ở Colab này. Không chọn model dựa trên test.


In [ ]:
OUTPUT=ROOT/'bundles'/f'grace-h3v2-{PHASE}-output.zip'
info=create_bundle(ROOT,OUTPUT,'output',PHASE)
print(json.dumps(info,indent=2))
exported=save_transfer(OUTPUT)
print('Output:',exported)
print('Quay về full-pipeline.ipynb local, đặt file vào bundles/ và Run All lại.')
if DOWNLOAD_OUTPUT:
    from google.colab import files
    files.download(str(OUTPUT))
